# Relation-specific error analysis.
This notebook focuses on identifying errors which are specific to some relations.

Moreover, we investigate whether the model can correctly identify the correct relation out of those which may be similar. To identify these groups we use relations which are pointed out as similar in the annotation guidelines (including inverse relations like `org:founded` and `org:founded_by`), along with the understanding of the language. 

The analysis on some of the most interesting groups is reported, as including all of them would be challenging (all 42 relations!)

As a summary of the observed insights we noticed that:
- Majority of misclassifications involve some relations being wrongly classified as `no_relation` or the opposite occuring.
- There are some errors which are caused by wrong subject or object type. For example some `org` are assigned `pos`, so the model is unable to correctly identify the relations.

In [1]:
import pandas as pd
import json
import os, sys
import numpy as np
import matplotlib.pyplot as plt

from utils import scorer

In [2]:
def print_sentence_relation_subj_obj(data, print_text="predicted"):
    for i, sentence in enumerate(data["token"]):
        print(f"{print_text}: {data.iloc[i][print_text]} (id: {data.iloc[i]['id']})")
        print(
            f"Subject: {data.iloc[i]['token'][data.iloc[i]['subj_start']:data.iloc[i]['subj_end']+1]}"
        )
        print(
            f"Object: {data.iloc[i]['token'][data.iloc[i]['obj_start']:data.iloc[i]['obj_end']+1]}"
        )
        print(f"{' '.join(sentence)}\n")

In [3]:
def summary_relation_FN(dataset, relation, summary_data, not_long=True):
    """
    Given a dataset and a relation, it outputs the sentences where that relation is the correct one and the predicted one is wrong (i.e. TP and FN for that relation).
    It provides some summaries (number of wrong relations, predicted relations, number of sentences with long distance between entities).
    Afterwards, it prints the sentences along with the entities and the predicted relations.
    If not_long is True, it only prints the sentences where the distance between entities is NOT long
    """
    data_relation = test_data[dataset["relation"] == relation]
    print(f"Number of sentences with relation {relation} : {len(data_relation)}")

    data_relation_wrong = data_relation[
        data_relation["relation"] != data_relation["predicted"]
    ]
    print(
        f"Number of sentences with wrong relation {relation} : {len(data_relation_wrong)}"
    )

    data_relation_wrong_not_long = data_relation_wrong[
        data_relation_wrong["entity_distance_categorical"] != "long"
    ]
    print(
        f"Of those sentences, {len(data_relation_wrong) - len(data_relation_wrong_not_long)} have long distance between entities"
    )

    wrong_relations_summary = data_relation_wrong["predicted"].value_counts()
    print(
        f"\nSummary of relations assigned instead of {relation}:\n{wrong_relations_summary}\n"
    )

    print(f"List of (NOT LONG) sentences with relation {relation} wrongly predicted")
    print_sentence_relation_subj_obj(
        data_relation_wrong_not_long if not_long else data_relation_wrong,
        print_text="predicted",
    )

In [4]:
def summary_relation_FP(dataset, relation, summary_data, not_long=True):
    """
    Given a dataset and a relation, it outputs the sentences the model outputs the relation while the correct one is different (i.e. TP and FP for that relation).
    It provides some summaries (number of wrong relations, predicted relations, number of sentences with long distance between entities).
    Afterwards, it prints the sentences along with the entities and the predicted relations.
    If not_long is True, it only prints the sentences where the distance between entities is NOT long
    """
    data_relation = test_data[dataset["predicted"] == relation]
    print(
        f"Number of sentences where the model predicts relation {relation} : {len(data_relation)}"
    )

    data_relation_wrong = data_relation[data_relation["relation"] != relation]
    print(
        f"Number of sentences where the model outputs {relation} but the true one is a different one: {len(data_relation_wrong)}"
    )

    data_relation_wrong_not_long = data_relation_wrong[
        data_relation_wrong["entity_distance_categorical"] != "long"
    ]
    print(
        f"Of those sentences, {len(data_relation_wrong) - len(data_relation_wrong_not_long)} have long distance between entities"
    )

    wrong_relations_summary = data_relation_wrong["relation"].value_counts()
    print(
        f"\nSummary of true relations to which {relation} is wrongly assigned:\n{wrong_relations_summary}\n"
    )

    print(f"List of (NOT LONG) sentences with relation {relation} assigned wrongly")
    print_sentence_relation_subj_obj(
        data_relation_wrong_not_long if not_long else data_relation_wrong,
        print_text="relation",
    )

Read the data

In [5]:
tacrev_results = pd.read_csv("./evaluation/errors_TACREV_model.txt", sep="\t")
summary_data = scorer.score(
    tacrev_results["Gold Label"],
    tacrev_results["Prediction"],
    verbose=True,
    return_data=True,
).rename(
    columns={
        "F1": "Tacred_F1",
        "Precision": "Tacred_precision",
        "Recall": "Tacred_recall",
    }
)
summary_data.rename(
    columns={
        "Relation": "relation",
        "Tacred_precision": "Tacrev_precision",
        "Tacred_recall": "Tacrev_recall",
        "Tacred_F1": "Tacrev_F1",
    },
    inplace=True,
)

Per-relation statistics:
org:alternate_names                  P:  95.98%  R:  87.76%  F1:  91.68%  #: 245
org:city_of_headquarters             P:  82.98%  R:  87.64%  F1:  85.25%  #: 89
org:country_of_headquarters          P:  91.43%  R:  34.78%  F1:  50.39%  #: 92
org:dissolved                        P: 100.00%  R:   0.00%  F1:   0.00%  #: 1
org:founded                          P:  86.84%  R:  89.19%  F1:  88.00%  #: 37
org:founded_by                       P:  80.56%  R:  38.16%  F1:  51.79%  #: 76
org:member_of                        P: 100.00%  R:   0.00%  F1:   0.00%  #: 4
org:members                          P:   0.00%  R:   0.00%  F1:   0.00%  #: 16
org:number_of_employees/members      P: 100.00%  R:  42.86%  F1:  60.00%  #: 14
org:parents                          P:  32.00%  R:  14.04%  F1:  19.51%  #: 57
org:political/religious_affiliation  P:  28.00%  R:  70.00%  F1:  40.00%  #: 10
org:shareholders                     P:   0.00%  R:   0.00%  F1:   0.00%  #: 3
org:stateorprovin

In [6]:
with open("../../../data/tacred/json/train.json", "r") as file:
    data_train = json.load(file)
training_data = pd.json_normalize(data_train)

# Count occurrences of each relation in the training set and include them in the summary
relation_counts = training_data["relation"].value_counts()
train_counts_df = relation_counts.reset_index()
train_counts_df.rename(columns={"count": "train_counts"}, inplace=True)
summary_data = pd.merge(summary_data, train_counts_df)

In [7]:
with open("../../../data/tacrev/json/test.json", "r") as file:
    test_data = json.load(file)
test_data = pd.json_normalize(test_data)

In [8]:
test_data["predicted"] = tacrev_results["Prediction"]
test_data["entity_distance"] = [
    (test_data["subj_start"][i] - test_data["obj_end"][i])
    if test_data["subj_start"][i] > test_data["obj_end"][i]
    else (test_data["obj_start"][i] - test_data["subj_end"][i])
    for i in range(len(test_data))
]
q1_ed = test_data["entity_distance"].quantile(0.3333)
q2_ed = test_data["entity_distance"].quantile(0.6666)
test_data["entity_distance_categorical"] = [
    "short"
    if test_data["entity_distance"][i] < q1_ed
    else ["long" if test_data["entity_distance"][i] > q2_ed else "medium"][0]
    for i in range(len(test_data))
]

- - - 
## ORG: Relations
### Focus on the family of relations relative to membership and relations with other companies/organisations
From these family we consider these types of relations, as they are mentioned as  in the slot annotation guidelines.
- `org:member_of` and `org:member` (inverse relations)
- `org:parents` and `org:subsidiaries` (inverse relations)
- `org:shareholders`

First of all extract statistics for these relationships:\
We can see that some appear only a few times in the test set, so it would not be correct to draw general conclusions on the model performance on those. These include `org:member_of` and `org:shareholders`.

In [9]:
membership_relations = [
    "org:member_of",
    "org:members",
    "org:parents",
    "org:subsidiaries",
    "org:shareholders",
]
filtered_summary_data = summary_data[
    summary_data["relation"].isin(membership_relations)
].sort_values("Tacrev_F1", ascending=False)
filtered_summary_data

,relation,Tacrev_precision,Tacrev_recall,Tacrev_F1,test_counts,train_counts
13,org:subsidiaries,0.486486,0.580645,0.529412,31,278
9,org:parents,0.320000,0.140351,0.195122,57,268
6,org:member_of,1.000000,0.000000,0.000000,4,113
7,org:members,0.000000,0.000000,0.000000,16,155
11,org:shareholders,0.000000,0.000000,0.000000,3,70


Focus on relation: `org:members`\
First we consider instances where the true relation was `org:members` and a different relation was predicted while the second cell contains instances of other relations where `org:members` was predicted wrongly.

In [10]:
summary_relation_FN(test_data, "org:members", summary_data)

Number of sentences with relation org:members : 16
Number of sentences with wrong relation org:members : 16
Of those sentences, 9 have long distance between entities

Summary of relations assigned instead of org:members:
predicted
no_relation         14
org:subsidiaries     2
Name: count, dtype: int64

List of (NOT LONG) sentences with relation org:members wrongly predicted
predicted: no_relation (id: 098f6e00a8a07ffb432f)
Subject: ['Organization', 'of', 'Asia', '-', 'Pacific', 'News', 'Agencies']
Object: ['Azerbaijani', 'wire', 'service']
Trend , an Azerbaijani wire service , on Thursday became a full member of the Organization of Asia - Pacific News Agencies -LRB- OANA -RRB- .

predicted: no_relation (id: 098f6e00a8c213b31b91)
Subject: ['Pacific', 'Asia', 'Travel', 'Association']
Object: ['Taiwan']
The fair was organized by the Pacific Asia Travel Association , of which Taiwan is a senior member .

predicted: no_relation (id: 098f6e00a84925a504b7)
Subject: ['Central', 'American', 'Pa

In [11]:
summary_relation_FP(test_data, "org:members", summary_data)

Number of sentences where the model predicts relation org:members : 2
Number of sentences where the model outputs org:members but the true one is a different one: 2
Of those sentences, 0 have long distance between entities

Summary of true relations to which org:members is wrongly assigned:
relation
no_relation         1
org:subsidiaries    1
Name: count, dtype: int64

List of (NOT LONG) sentences with relation org:members assigned wrongly
relation: no_relation (id: 098f665fb973ab8f29c2)
Subject: ['Organisation', 'of', 'Asia-Pacific', 'News', 'Agencies']
Object: ['general', 'assembly']
Earlier this week Jakarta hosted the general assembly of the Organisation of Asia-Pacific News Agencies , during which Antara signed cooperation agreements with its Turkish and Azerbaijani counterparts .

relation: org:subsidiaries (id: 098f665fb9ff389142c6)
Subject: ['Organisation', 'of', 'Asia-Pacific', 'News', 'Agencies']
Object: ['general', 'assembly']
The general assembly of the Organisation of Asia

It appears that the model tends to predict `org:subsidiaries` rather than the correct `org:member_of`. In all other cases the model predicts `no_relation` which appears to be the most common error.

Now we move to the other two members of this group of relations, i.e. `org:subsidiaries` and afterwards `org:parents`

In [12]:
summary_relation_FN(test_data, "org:subsidiaries", summary_data)

Number of sentences with relation org:subsidiaries : 31
Number of sentences with wrong relation org:subsidiaries : 13
Of those sentences, 3 have long distance between entities

Summary of relations assigned instead of org:subsidiaries:
predicted
no_relation    10
org:parents     2
org:members     1
Name: count, dtype: int64

List of (NOT LONG) sentences with relation org:subsidiaries wrongly predicted
predicted: org:members (id: 098f665fb9ff389142c6)
Subject: ['Organisation', 'of', 'Asia-Pacific', 'News', 'Agencies']
Object: ['general', 'assembly']
The general assembly of the Organisation of Asia-Pacific News Agencies -LRB- OANA -RRB- is seeking to boost the quality of the 40 news agencies across 33 countries that comprise it , said incoming OANA head and chief of Indonesia 's state-run Antara news agency Ahmad Mukhlis Yusuf .

predicted: no_relation (id: 098f6c74c5248883a246)
Subject: ['Nuclear', 'Decommissioning', 'Authority']
Object: ['Sellafield']
The consortium , Nuclear Managemen

In [13]:
summary_relation_FP(test_data, "org:subsidiaries", summary_data)

Number of sentences where the model predicts relation org:subsidiaries : 37
Number of sentences where the model outputs org:subsidiaries but the true one is a different one: 19
Of those sentences, 2 have long distance between entities

Summary of true relations to which org:subsidiaries is wrongly assigned:
relation
no_relation      11
org:parents       5
org:members       2
org:member_of     1
Name: count, dtype: int64

List of (NOT LONG) sentences with relation org:subsidiaries assigned wrongly
relation: no_relation (id: 098f665fb9519aeaeb31)
Subject: ['Corporate', 'Library']
Object: ['Bank', 'of', 'America']
Paul Hodgson , research associate at the Corporate Library , a corporate governance research firm , criticized Bank of America for agreeing to the terms of Sambol 's exit deal .

relation: org:member_of (id: 098f6784e78abd5c3e0c)
Subject: ['ALICO']
Object: ['AIG']
ALICO , a member company of AIG is looking for one J2EE developer to help with a significant re-engineering project 

A similar behaviour is observed, in this case it appears that in many intances, the errors are due to the relations `org:subsidiaries` and `org:parents` are misclassified by swapping one with the other; note that these two relations are one the reverse of the other.


> Also note that in sentence (098f6c74c51d0732d3bd) the model appears to be correct, while the label appears to be wrong.

predicted: `org:parents` (id: 098f6c74c51d0732d3bd)\
label: `org:subsidiaries`\
Subject: ['Semen', 'Gresik']\
Object: ['Semen', 'Padang']\
However , the government let the deal expire in December 2001 amid protests from local politicians and workers at a Semen Gresik unit , Semen Padang in West Sumatra .

This points to some errors still being present in the dataset, as only part of it was revisited.


It appears that as expected on most instances the model either wrongly assigns no relation to sentences where the relation is of a different type or vice-versa it assigns the specific relation on no relation instances.

Whenever the error is due to a specific sentence, it mixes relations which are all (or almost) within the same group as defined above. 

- - - 
### Focus on the family of relations relative to location of headquarters of a company.

These are:
- `org:city_of_headquarters`
- `org:country_of_headquarters`
- `org:stateorprovince_of_headquarters`

On these the model on average achieves a quite high precision and recall, with a smaller recall on `org:country_of_headquarters`.

In [14]:
membership_relations = [
    "org:city_of_headquarters",
    "org:country_of_headquarters",
    "org:stateorprovince_of_headquarters",
]
filtered_summary_data = summary_data[
    summary_data["relation"].isin(membership_relations)
].sort_values("Tacrev_F1", ascending=False)
filtered_summary_data

,relation,Tacrev_precision,Tacrev_recall,Tacrev_F1,test_counts,train_counts
1,org:city_of_headquarters,0.829787,0.876404,0.852459,89,357
12,org:stateorprovince_of_headquarters,0.745098,0.745098,0.745098,51,221
2,org:country_of_headquarters,0.914286,0.347826,0.503937,92,438


In [15]:
summary_relation_FN(test_data, "org:city_of_headquarters", summary_data)

Number of sentences with relation org:city_of_headquarters : 89
Number of sentences with wrong relation org:city_of_headquarters : 11
Of those sentences, 1 have long distance between entities

Summary of relations assigned instead of org:city_of_headquarters:
predicted
no_relation                10
per:cities_of_residence     1
Name: count, dtype: int64

List of (NOT LONG) sentences with relation org:city_of_headquarters wrongly predicted
predicted: no_relation (id: 098f65158dde1a8effc2)
Subject: ['Greenberg', 'Smoked', 'Turkey']
Object: ['Tyler']
TURKEY-VENDOR -LRB- Tyler , Texas -RRB- -- By the time Sam Greenberg closes Greenberg Smoked Turkey in Tyler for the holiday season on Dec 24 and hands each of his 200 employees a free bird , more than 200,000 turkeys will have emerged from the company 's 20 brick-lined , hardwood-fired pit houses .

predicted: no_relation (id: 098f65158d7ae6af12ee)
Subject: ['Countrywide']
Object: ['Calabasas']
Continued 1 | 2 | 3 Next > Calabasas , Californ

We note something interesting from the output above. Consider sentence with id 098f60af8f4f637db90f, where the model assigns relationship `per:cities_of_residence`, which appears very different from the true relation of `org:city_of_headquarters`.

Subject: [Sycamore]\
Object: [Chelmsford]\
**Sycamore** , a **Chelmsford** company that makes optical-networking equipment , is just one company among more than 200 struggling to investigate and ultimately explain evidence the dates on stock option awards were altered or otherwise rigged to make the perks more valuable .

However this appears to be caused by the fact in the test set, the entity Sycamore is wrongly assigned subject type `PER`, while it is clearly an organisation.

In [16]:
summary_relation_FN(test_data, "org:country_of_headquarters", summary_data)

Number of sentences with relation org:country_of_headquarters : 92
Number of sentences with wrong relation org:country_of_headquarters : 60
Of those sentences, 28 have long distance between entities

Summary of relations assigned instead of org:country_of_headquarters:
predicted
no_relation    56
per:origin      4
Name: count, dtype: int64

List of (NOT LONG) sentences with relation org:country_of_headquarters wrongly predicted
predicted: no_relation (id: 098f6179d3bb2b93c59d)
Subject: ['Countrywide', 'Financial']
Object: ['US']
An optimistic outlook from troubled US mortgage giant Countrywide Financial on Friday , coupled with expectations for a US rate cut on Wednesday , raised hopes that the fallout from the US housing woes is easing , dealers said .

predicted: no_relation (id: 098f6179d38875ffc221)
Subject: ['National', 'Congress', 'of', 'American', 'Indians']
Object: ['Washington', ',', 'DC']
Iroquois passport dispute raises sovereignty issue The National Congress of American Ind

Also in the sentence with id 098f6379356a57b0663a, a similar error is observed.

Subject: ['Alessi']\
Object: ['Italian']\
Alberto Alessi , president of the **Italian** design firm **Alessi** , will speak Nov. 18 at the museum about the role his family 's business has played in bringing creative and functional design to the world market .

Also here, the error appears to be caused by the fact that the company Alessi has associated subject type `PER`, instead of the correct one which should be `ORG`.

In [17]:
summary_relation_FP(test_data, "org:city_of_headquarters", summary_data)

Number of sentences where the model predicts relation org:city_of_headquarters : 94
Number of sentences where the model outputs org:city_of_headquarters but the true one is a different one: 16
Of those sentences, 4 have long distance between entities

Summary of true relations to which org:city_of_headquarters is wrongly assigned:
relation
no_relation                            15
org:stateorprovince_of_headquarters     1
Name: count, dtype: int64

List of (NOT LONG) sentences with relation org:city_of_headquarters assigned wrongly
relation: no_relation (id: 098f665fb9089e6ebb71)
Subject: ['Norris', 'Church', 'Mailer']
Object: ['Brooklyn']
Mailer resided in Provincetown , Massachusetts , with his wife of 33 years , Norris Church Mailer , and maintained an apartment in Brooklyn , New York .

relation: org:stateorprovince_of_headquarters (id: 098f65158d567ded4b2a)
Subject: ['National', 'Congress', 'of', 'American', 'Indians']
Object: ['D.C.']
The National Congress of American Indians , b

Finally, a similar array of issues arise when D.C. is the object of some relation. 

Consider for example the sentence with id 098f65158d567ded4b2a where the correct relation is `org:stateorprovince_of_headquarters`, while the model predicts `org:city_of_headquarters`.

Subject: ['National', 'Congress', 'of', 'American', 'Indians']\
Object: ['D.C.']\
The **National Congress of American Indians** , based in Washington , **D.C.** , has advocated on behalf of the lacrosse team , urging British officials to allow the members entry into England on their Iroquois-issued passports .

The reason appears to be that D.C. is assigned the object type `LOCATION`, rather than the more specific ones which are available like `STATE` (the correct one) or `CITY`. Due to the masking performed by the model, the model only has the object type to perform the prediction, and as such it has no way to reconstruct whether it is a city or a state.
